# Phase 5 — EDA: Prometheus Metrics for ML Anomaly Detection

This notebook performs **Exploratory Data Analysis** on the CSV collected by `infra/ml-data-collector/prometheus_exporter.py`.

**Data schema (one row = one service × one scrape):**
| Column | Description |
|---|---|
| `timestamp` | ISO-8601 UTC scrape time |
| `job` | Prometheus job label (`agent1-inventory`, etc.) |
| `service_name` | Short name (`inventory`, `order`, …) |
| `cluster` | `agent1` / `agent2` / `manager` |
| `is_down` | 1 = service unreachable, 0 = healthy |
| `memory_used_bytes` | JVM heap bytes |
| `request_rate` | HTTP req/s (1-min rate) |
| `error_rate` | 5xx req/s (1-min rate) |
| `cpu_usage` | 0–1 system CPU fraction |
| `feign_failures` | OpenFeign call failures (1-min increase) |
| `label` | Ground-truth scenario: `normal` / `degraded` / `down` |

**Services covered:** 16 total — agent1 ×4, agent2 ×4, manager ×8

In [ ]:
## ── 1. Imports & Configuration ───────────────────────────────────────────────
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

pd.set_option("display.max_columns", 20)
pd.set_option("display.float_format", "{:.4f}".format)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})

# ── Paths ─────────────────────────────────────────────────────────────────────
NOTEBOOK_DIR = Path(".")
CSV_PATH     = NOTEBOOK_DIR / "training_data.csv"
SCHEMA_PATH  = NOTEBOOK_DIR / "sample_schema.json"

# ── Metadata ──────────────────────────────────────────────────────────────────
JOB_META = {
    "agent1-inventory":            ("inventory",            "agent1"),
    "agent1-order":                ("order",               "agent1"),
    "agent1-revenue":              ("revenue",             "agent1"),
    "agent1-product":              ("product",             "agent1"),
    "agent2-inventory":            ("inventory",           "agent2"),
    "agent2-order":                ("order",               "agent2"),
    "agent2-revenue":              ("revenue",             "agent2"),
    "agent2-product":              ("product",             "agent2"),
    "manager-data-aggregation":    ("data-aggregation",    "manager"),
    "manager-central-analytics":   ("central-analytics",   "manager"),
    "manager-alert":               ("alert",               "manager"),
    "manager-automation-action":   ("automation-action",   "manager"),
    "manager-report":              ("report",              "manager"),
    "manager-master-data":         ("master-data",         "manager"),
    "manager-ml-bridge":           ("ml-bridge",           "manager"),
    "manager-central-api-gateway": ("central-api-gateway", "manager"),
}

NUMERIC_COLS  = ["memory_used_bytes", "request_rate", "error_rate",
                 "cpu_usage", "feign_failures", "is_down"]
LABEL_COLORS  = {"normal": "#2ca02c", "degraded": "#ff7f0e", "down": "#d62728"}
LABEL_ORDER   = ["normal", "degraded", "down"]

print("Imports OK — CSV target:", CSV_PATH.resolve())
print("CSV exists:", CSV_PATH.exists())

In [ ]:
## ── 2. Load Data (or generate synthetic sample if CSV not ready yet) ─────────
from datetime import datetime, timezone, timedelta
import random

def _synthetic_df(n_services: int = 16, polls_per_min: int = 4,
                  scenario_minutes: tuple = (10, 5, 5)) -> pd.DataFrame:
    """Generate a minimal synthetic dataset that mirrors the real CSV schema."""
    rng  = np.random.default_rng(42)
    rows = []
    jobs = list(JOB_META.keys())
    t0   = datetime.now(timezone.utc)

    scenario_cfg = [
        ("normal",   scenario_minutes[0], False, False),
        ("degraded", scenario_minutes[1], True,  False),
        ("down",     scenario_minutes[2], True,  True),
    ]

    for label, dur_min, some_down, all_down in scenario_cfg:
        for tick in range(dur_min * polls_per_min):
            ts = t0 + timedelta(seconds=tick * (60 // polls_per_min))
            for job in jobs:
                svc, cluster = JOB_META[job]
                is_agent1 = cluster == "agent1"
                # Simulate which services go down
                if all_down and is_agent1:
                    is_down = 1
                elif some_down and is_agent1 and job in ("agent1-inventory", "agent1-order"):
                    is_down = 1
                else:
                    is_down = 0

                base_mem = rng.normal(280_000_000, 20_000_000)
                rows.append({
                    "timestamp":         ts.isoformat(),
                    "job":               job,
                    "service_name":      svc,
                    "cluster":           cluster,
                    "is_down":           is_down,
                    "memory_used_bytes": max(0, base_mem) if not is_down else 0,
                    "request_rate":      max(0, rng.normal(8, 2)) if not is_down else 0,
                    "error_rate":        abs(rng.normal(0.05, 0.03)) if label in ("degraded","down") else abs(rng.normal(0.001, 0.001)),
                    "cpu_usage":         max(0, min(1, rng.normal(0.35, 0.1))) if not is_down else 0,
                    "feign_failures":    abs(rng.normal(0.5, 0.3)) if label != "normal" else 0,
                    "label":             label,
                })
        t0 += timedelta(minutes=dur_min)

    return pd.DataFrame(rows)


if CSV_PATH.exists():
    df = pd.read_csv(CSV_PATH, parse_dates=["timestamp"])
    print(f"Loaded REAL data from {CSV_PATH}  →  {len(df):,} rows")
else:
    print(f"[INFO] {CSV_PATH} not found — generating synthetic sample data for preview")
    df = _synthetic_df()
    print(f"Generated synthetic data  →  {len(df):,} rows")

# Ensure correct dtypes
df["timestamp"]      = pd.to_datetime(df["timestamp"], utc=True, errors="coerce")
df["is_down"]        = df["is_down"].astype(int)
for col in NUMERIC_COLS:
    df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)

print(f"Shape: {df.shape}   |   Labels: {df['label'].value_counts().to_dict()}")

In [ ]:
## ── 3. Data Preview ──────────────────────────────────────────────────────────
print("=== HEAD ===")
display(df.head(10))

print("\n=== DTYPES & NON-NULL ===")
df.info(show_counts=True)

print("\n=== DESCRIBE ===")
display(df[NUMERIC_COLS].describe().round(4))

print("\n=== NULL CHECK ===")
print(df.isnull().sum())

print("\n=== TIME RANGE ===")
print("Start:", df["timestamp"].min())
print("End  :", df["timestamp"].max())
print("Span :", df["timestamp"].max() - df["timestamp"].min())

print("\n=== SERVICES ===")
print(f"Unique jobs    : {df['job'].nunique()}")
print(f"Unique clusters: {df['cluster'].unique()}")

## 4 — Time Series: Each Metric Over Time

One subplot per metric. Lines are coloured by scenario label so anomalies are clearly visible.
We aggregate across all services (mean per timestamp × label) to reduce noise.

In [ ]:
## ── 4. Time Series Plots ──────────────────────────────────────────────────────
PLOT_METRICS = [
    ("memory_used_bytes", "Memory Used (bytes)",  "JVM Heap Used"),
    ("request_rate",      "Requests / s",          "HTTP Request Rate (1-min)"),
    ("error_rate",        "5xx Errors / s",        "HTTP Error Rate (1-min)"),
    ("cpu_usage",         "CPU Fraction (0–1)",    "System CPU Usage"),
    ("feign_failures",    "Failures / min",        "OpenFeign Call Failures"),
    ("is_down",           "Is-Down Flag",          "Service Down Indicator"),
]

# Aggregate: mean per (timestamp, label) across all services
agg = (df.groupby(["timestamp", "label"])[NUMERIC_COLS]
         .mean()
         .reset_index()
         .sort_values("timestamp"))

fig, axes = plt.subplots(len(PLOT_METRICS), 1, figsize=(16, 3.5 * len(PLOT_METRICS)),
                         sharex=True)
fig.suptitle("Prometheus Metrics Over Time — Coloured by Scenario Label",
             fontsize=14, fontweight="bold", y=1.002)

for ax, (col, ylabel, title) in zip(axes, PLOT_METRICS):
    for label in LABEL_ORDER:
        sub = agg[agg["label"] == label]
        ax.plot(sub["timestamp"], sub[col],
                label=label, color=LABEL_COLORS[label], linewidth=1.4, alpha=0.85)
    ax.set_ylabel(ylabel, fontsize=9)
    ax.set_title(title, fontsize=10, fontweight="bold")
    ax.legend(loc="upper right", fontsize=8)

axes[-1].xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
axes[-1].xaxis.set_major_locator(mdates.AutoDateLocator())
axes[-1].set_xlabel("Time (UTC)", fontsize=9)
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

## 5 — Correlation Heatmap

Shows Pearson correlations between all numeric features.
High correlations (|r| > 0.7) may indicate redundant features that can be dropped before training.

In [ ]:
## ── 5. Correlation Heatmap ────────────────────────────────────────────────────
corr = df[NUMERIC_COLS].corr()

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0,
            vmin=-1, vmax=1, square=True, linewidths=0.5, ax=ax)
ax.set_title("Feature Correlation Matrix", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

# Report strong correlations
threshold = 0.7
strong = [(r, c, corr.loc[r, c])
          for r in corr.index for c in corr.columns
          if r < c and abs(corr.loc[r, c]) >= threshold]
if strong:
    print(f"\nStrong correlations (|r| ≥ {threshold}):")
    for r, c, v in sorted(strong, key=lambda x: -abs(x[2])):
        print(f"  {r:25s} ↔ {c:25s}  r = {v:.3f}")
else:
    print(f"No feature pairs with |r| ≥ {threshold}")

## 6 — Class Distribution

Check for class imbalance.  
If any class ratio exceeds **3 : 1**, consider using `class_weight="balanced"` or SMOTE when training.

In [ ]:
## ── 6. Class Distribution ────────────────────────────────────────────────────
counts = df["label"].value_counts().reindex(LABEL_ORDER, fill_value=0)
pct    = (counts / counts.sum() * 100).round(1)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Bar chart
axes[0].bar(counts.index, counts.values,
            color=[LABEL_COLORS[l] for l in counts.index], edgecolor="white")
for i, (v, p) in enumerate(zip(counts.values, pct.values)):
    axes[0].text(i, v + counts.max() * 0.02, f"{v:,}\n({p}%)",
                 ha="center", va="bottom", fontsize=9)
axes[0].set_xlabel("Scenario Label", fontsize=10)
axes[0].set_ylabel("Row Count",      fontsize=10)
axes[0].set_title("Class Distribution — Row Counts", fontsize=11, fontweight="bold")

# Pie chart
axes[1].pie(counts.values, labels=counts.index, autopct="%1.1f%%",
            colors=[LABEL_COLORS[l] for l in counts.index],
            startangle=90, wedgeprops={"edgecolor": "white", "linewidth": 1})
axes[1].set_title("Class Distribution — Proportion", fontsize=11, fontweight="bold")

plt.tight_layout()
plt.show()

# Imbalance check
max_ratio = counts.max() / max(counts.min(), 1)
if max_ratio > 3:
    print(f"⚠️  Class imbalance detected (max/min ratio = {max_ratio:.1f}x).")
    print("   Consider: class_weight='balanced' or SMOTE (imbalanced-learn).")
else:
    print(f"✅ Classes are reasonably balanced (max/min ratio = {max_ratio:.1f}x).")

## 7 — Feature Statistics & Boxplots by Label

Mean / std / min / max of every feature broken down by scenario label.
The boxplots show how well each feature separates the three classes.

In [ ]:
## ── 7. Feature Statistics & Boxplots ────────────────────────────────────────
# Grouped statistics
stats = df.groupby("label")[NUMERIC_COLS].agg(["mean", "std", "min", "max"]).round(4)
print("=== Feature Statistics by Label ===")
display(stats)

# Boxplots
n_cols = 3
n_rows = int(np.ceil(len(NUMERIC_COLS) / n_cols))
fig, axes = plt.subplots(n_rows, n_cols,
                          figsize=(5 * n_cols, 4 * n_rows), squeeze=False)
axes_flat = axes.flatten()

for i, col in enumerate(NUMERIC_COLS):
    ax = axes_flat[i]
    data_by_label = [df.loc[df["label"] == lbl, col].values for lbl in LABEL_ORDER]
    bp = ax.boxplot(data_by_label, labels=LABEL_ORDER, patch_artist=True,
                    medianprops={"color": "black", "linewidth": 1.5})
    for patch, lbl in zip(bp["boxes"], LABEL_ORDER):
        patch.set_facecolor(LABEL_COLORS[lbl])
        patch.set_alpha(0.7)
    ax.set_title(col, fontsize=10, fontweight="bold")
    ax.set_xlabel("Label", fontsize=9)

# Hide unused axes
for i in range(len(NUMERIC_COLS), len(axes_flat)):
    axes_flat[i].set_visible(False)

fig.suptitle("Feature Distributions by Scenario Label", fontsize=13,
             fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

## 8 — Per-Service is_down Heatmap

Which services were down and in which scenario?  
Useful for verifying the simulation scripts ran correctly.

In [ ]:
## ── 8. Per-Service is_down Heatmap ───────────────────────────────────────────
pivot = (df.groupby(["job", "label"])["is_down"]
           .mean()
           .unstack("label")
           .reindex(columns=LABEL_ORDER, fill_value=0)
           .sort_index())

fig, ax = plt.subplots(figsize=(7, max(6, len(pivot) * 0.45)))
sns.heatmap(pivot, annot=True, fmt=".2f", cmap="Reds",
            vmin=0, vmax=1, linewidths=0.4, ax=ax,
            cbar_kws={"label": "Mean is_down (0=up, 1=down)"})
ax.set_title("Average is_down Fraction per Service & Scenario",
             fontsize=12, fontweight="bold")
ax.set_xlabel("Scenario Label", fontsize=10)
ax.set_ylabel("Prometheus Job",  fontsize=10)
ax.tick_params(axis="x", rotation=0)
ax.tick_params(axis="y", rotation=0)
plt.tight_layout()
plt.show()

# Sanity check: agent1 services should be down in "down" scenario
agent1_down = pivot.loc[pivot.index.str.startswith("agent1"), "down"]
print("agent1 services — mean is_down in 'down' scenario:")
print(agent1_down.to_string())

## 9 — Export Validated CSV & Schema

Save the DataFrame to `training_data.csv` and write a `sample_schema.json` that Phase 6 (ML training) will use to validate in-coming data.

In [ ]:
## ── 9. Export & Schema ───────────────────────────────────────────────────────
# Save CSV (overwrites if this was synthetic, or is a clean pass of real data)
out_path = CSV_PATH
df.to_csv(out_path, index=False)
print(f"Saved {len(df):,} rows → {out_path.resolve()}")

# Schema JSON for Phase 6
schema = {
    "version": "1.0",
    "columns": {
        "timestamp":         {"dtype": "datetime64[ns, UTC]", "description": "Scrape time (ISO-8601 UTC)"},
        "job":               {"dtype": "str",    "description": "Prometheus job label",        "example": "agent1-inventory"},
        "service_name":      {"dtype": "str",    "description": "Short service name",           "example": "inventory"},
        "cluster":           {"dtype": "str",    "description": "Cluster name",                 "values": ["agent1","agent2","manager"]},
        "is_down":           {"dtype": "int",    "description": "1=down, 0=up",                 "range": [0, 1]},
        "memory_used_bytes": {"dtype": "float",  "description": "JVM heap used (bytes)",        "range": [0, None]},
        "request_rate":      {"dtype": "float",  "description": "HTTP req/s (1-min rate)",      "range": [0, None]},
        "error_rate":        {"dtype": "float",  "description": "5xx req/s (1-min rate)",       "range": [0, None]},
        "cpu_usage":         {"dtype": "float",  "description": "System CPU fraction",          "range": [0, 1]},
        "feign_failures":    {"dtype": "float",  "description": "Feign failures/min (1-min inc.)","range": [0, None]},
        "label":             {"dtype": "str",    "description": "Ground-truth scenario",        "values": ["normal","degraded","down"]},
    },
    "stats": {
        "total_rows":   len(df),
        "label_counts": df["label"].value_counts().to_dict(),
        "time_range":   [str(df["timestamp"].min()), str(df["timestamp"].max())],
        "services":     sorted(df["job"].unique().tolist()),
    }
}

SCHEMA_PATH.write_text(json.dumps(schema, indent=2), encoding="utf-8")
print(f"Schema written → {SCHEMA_PATH.resolve()}")

# Final validation
assert len(df) >= 1000, f"Expected ≥1000 rows, got {len(df)}"
print(f"\n✅ Validation passed: {len(df):,} rows, {df['label'].nunique()} labels, "
      f"{df['job'].nunique()} services")
print("\n=== Final Stats ===")
print(df["label"].value_counts().to_frame("count"))